## **PROJET TINDER - EDA**

### **1. Compréhension du dataset**

**Speed Dating Dataset**

=============================================================================

**Objectif** : 
- Comprendre la structure des colonnes, 
- identifier les groupes de variables, 
- détecter les problèmes de qualité, 
- et préparer un dataset propre pour l'EDA.

=============================================================================


#### **Import et Configuration**

In [41]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
from plotly.subplots import make_subplots
 
# Style global Plotly
pio.templates.default = "plotly_white"
PALETTE = {
    "match":    "#1D9E75",   
    "no_match": "#E24B4A",   
    "male":     "#378ADD",   
    "female":   "#D4537E",   
    "neutral":  "#888780",   
    "purple":   "#7F77DD",
    "amber":    "#EF9F27",
    "coral":    "#D85A30",
}

PROJECT_ROOT = Path('../')
RAW_DATA_PATH = PROJECT_ROOT / 'data' 
OUTPUT_DATA_PATH = PROJECT_ROOT / 'outputs' / 'data'
print(RAW_DATA_PATH, OUTPUT_DATA_PATH)

../data ../outputs/data


In [42]:
# =============================================================================
# 1. CHARGEMENT DES DONNÉES
# =============================================================================
 
df = pd.read_csv(f"{RAW_DATA_PATH}/speed_dating_data.csv", encoding="latin-1")
 
print(f"Dataset chargé : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"Chaque ligne = une interaction (date) entre deux personnes")
print(f"Target principale : 'match' (1 = oui, 0 = non)")


Dataset chargé : 8,378 lignes x 195 colonnes
Chaque ligne = une interaction (date) entre deux personnes
Target principale : 'match' (1 = oui, 0 = non)


In [43]:
# =============================================================================
# 2. CARTOGRAPHIE DES COLONNES PAR GROUPE THÉMATIQUE
# =============================================================================
 
GROUPS = {
    "Identité": [
        "iid", "id", "gender", "idg", "condtn", "wave",
        "round", "position", "positin1", "order", "partner", "pid"
    ],
    "Démographie": [
        "age", "field", "field_cd", "undergra", "mn_sat", "tuition",
        "race", "imprace", "imprelig", "from", "zipcode", "income",
        "goal", "date", "go_out", "career", "career_c"
    ],
    "Intérêts personnels": [
        "sports", "tvsports", "exercise", "dining", "museums", "art",
        "hiking", "gaming", "clubbing", "reading", "tv", "theater",
        "movies", "concerts", "music", "shopping", "yoga"
    ],
    "T1 — Préférences déclarées (avant event)": [
        "attr1_1", "sinc1_1", "intel1_1", "fun1_1", "amb1_1", "shar1_1",
        "attr2_1", "sinc2_1", "intel2_1", "fun2_1", "amb2_1", "shar2_1",
        "attr3_1", "sinc3_1", "fun3_1", "intel3_1", "amb3_1",
        "attr4_1", "sinc4_1", "intel4_1", "fun4_1", "amb4_1", "shar4_1",
        "attr5_1", "sinc5_1", "intel5_1", "fun5_1", "amb5_1",
        "exphappy", "expnum"
    ],
    "Scorecard soirée (sujet → partenaire)": [
        "dec", "attr", "sinc", "intel", "fun", "amb", "shar", "like", "prob", "met"
    ],
    "Scorecard soirée (partenaire → sujet)": [
        "dec_o", "attr_o", "sinc_o", "intel_o", "fun_o", "amb_o",
        "shar_o", "like_o", "prob_o", "met_o"
    ],
    "Target + Contexte interaction": [
        "match", "int_corr", "samerace", "age_o", "race_o",
        "pf_o_att", "pf_o_sin", "pf_o_int", "pf_o_fun", "pf_o_amb", "pf_o_sha"
    ],
    "T1 mi-soirée (scorecard intermédiaire)": [
        "match_es",
        "attr1_s", "sinc1_s", "intel1_s", "fun1_s", "amb1_s", "shar1_s",
        "attr3_s", "sinc3_s", "intel3_s", "fun3_s", "amb3_s"
    ],
    "T2 — Lendemain (followup)": [
        "satis_2", "length", "numdat_2",
        "attr7_2", "sinc7_2", "intel7_2", "fun7_2", "amb7_2", "shar7_2",
        "attr1_2", "sinc1_2", "intel1_2", "fun1_2", "amb1_2", "shar1_2",
        "attr4_2", "sinc4_2", "intel4_2", "fun4_2", "amb4_2", "shar4_2",
        "attr2_2", "sinc2_2", "intel2_2", "fun2_2", "amb2_2", "shar2_2",
        "attr3_2", "sinc3_2", "intel3_2", "fun3_2", "amb3_2",
        "attr5_2", "sinc5_2", "intel5_2", "fun5_2", "amb5_2"
    ],
    "T3 — 3-4 semaines après": [
        "you_call", "them_cal", "date_3", "numdat_3", "num_in_3",
        "attr1_3", "sinc1_3", "intel1_3", "fun1_3", "amb1_3", "shar1_3",
        "attr7_3", "sinc7_3", "intel7_3", "fun7_3", "amb7_3", "shar7_3",
        "attr4_3", "sinc4_3", "intel4_3", "fun4_3", "amb4_3", "shar4_3",
        "attr2_3", "sinc2_3", "intel2_3", "fun2_3", "amb2_3", "shar2_3",
        "attr3_3", "sinc3_3", "intel3_3", "fun3_3", "amb3_3",
        "attr5_3", "sinc5_3", "intel5_3", "fun5_3", "amb5_3"
    ],
}

print("\n" + "─" * 65)
print("CARTOGRAPHIE DES 195 COLONNES PAR GROUPE")
print("─" * 65)
total_mapped = 0
for group, cols in GROUPS.items():
    existing = [c for c in cols if c in df.columns]
    total_mapped += len(existing)
    print(f"  {group:<48} {len(existing):>3} cols")
print(f"{'':>50}──────")
print(f"  {'Total colonnes mappées':<48} {total_mapped:>3}")
print(f"  {'Total colonnes dataset':<48} {df.shape[1]:>3}")


─────────────────────────────────────────────────────────────────
CARTOGRAPHIE DES 195 COLONNES PAR GROUPE
─────────────────────────────────────────────────────────────────
  Identité                                          12 cols
  Démographie                                       17 cols
  Intérêts personnels                               17 cols
  T1 — Préférences déclarées (avant event)          30 cols
  Scorecard soirée (sujet → partenaire)             10 cols
  Scorecard soirée (partenaire → sujet)             10 cols
  Target + Contexte interaction                     11 cols
  T1 mi-soirée (scorecard intermédiaire)            12 cols
  T2 — Lendemain (followup)                         37 cols
  T3 — 3-4 semaines après                           39 cols
                                                  ──────
  Total colonnes mappées                           195
  Total colonnes dataset                           195


In [44]:
# =============================================================================
# 3. ANALYSE DES VALEURS MANQUANTES PAR GROUPE
# =============================================================================
 
print("\n" + "─" * 65)
print("QUALITÉ DES DONNÉES — VALEURS MANQUANTES PAR GROUPE")
print("─" * 65)
 
missing_summary = []
for group, cols in GROUPS.items():
    existing = [c for c in cols if c in df.columns]
    if not existing:
        continue
    pct_missing = df[existing].isnull().mean().mean() * 100
    missing_summary.append({
        "Groupe": group,
        "Nb colonnes": len(existing),
        "% manquant moyen": pct_missing,
        "Qualité": "Excellente" if pct_missing < 5
                   else "Acceptable" if pct_missing < 30 # 30%
                   else "Très incomplet"
    })
    print(f"  {group:<48} {pct_missing:>6.1f}%  {missing_summary[-1]['Qualité']}")
 
# Visualisation heatmap des valeurs manquantes
cols_focus = (
    ["match", "gender", "wave"] +
    ["attr", "sinc", "intel", "fun", "amb", "shar", "like", "prob", "dec"] +
    ["attr_o", "sinc_o", "intel_o", "fun_o", "like_o", "dec_o"] +
    ["attr1_1", "sinc1_1", "intel1_1", "fun1_1", "amb1_1", "shar1_1"] +
    ["attr3_1", "sinc3_1", "fun3_1", "intel3_1"] +
    ["int_corr", "samerace", "age", "age_o", "goal", "go_out"] +
    ["attr7_2", "sinc7_2", "fun7_2", "satis_2"] +
    ["attr1_3", "date_3", "numdat_3"]
)
cols_focus = [c for c in cols_focus if c in df.columns]
 
missing_pct = (df[cols_focus].isnull().mean() * 100).reset_index()
missing_pct.columns = ["Colonne", "% Manquant"]
missing_pct["Couleur"] = missing_pct["% Manquant"].apply(
    lambda x: "< 5%" if x < 5 else "5-30%" if x < 30 else "> 30%"
)
 
fig_missing = px.bar(
    missing_pct,
    x="Colonne",
    y="% Manquant",
    color="Couleur",
    color_discrete_map={"< 5%": PALETTE["match"], "5-30%": PALETTE["amber"], "> 30%": PALETTE["no_match"]},
    title="Valeurs manquantes — colonnes clés pour l'analyse du match",
    labels={"% Manquant": "% valeurs manquantes"},
    height=450,
)
fig_missing.update_layout(
    xaxis_tickangle=-45,
    legend_title="Seuil",
    title_font_size=16,
    plot_bgcolor="white",
)
fig_missing.add_hline(y=5, line_dash="dot", line_color=PALETTE["match"],
                      annotation_text="Seuil 5%", annotation_position="right")
fig_missing.add_hline(y=30, line_dash="dot", line_color=PALETTE["amber"],
                      annotation_text="Seuil 30%", annotation_position="right")
fig_missing.show()


─────────────────────────────────────────────────────────────────
QUALITÉ DES DONNÉES — VALEURS MANQUANTES PAR GROUPE
─────────────────────────────────────────────────────────────────
  Identité                                            1.8%  Excellente
  Démographie                                        13.8%  Acceptable
  Intérêts personnels                                 0.9%  Excellente
  T1 — Préférences déclarées (avant event)           14.7%  Acceptable
  Scorecard soirée (sujet → partenaire)               4.6%  Excellente
  Scorecard soirée (partenaire → sujet)               4.7%  Excellente
  Target + Contexte interaction                       1.0%  Excellente
  T1 mi-soirée (scorecard intermédiaire)             48.5%  Très incomplet
  T2 — Lendemain (followup)                          33.1%  Très incomplet
  T3 — 3-4 semaines après                            64.9%  Très incomplet


In [45]:
# =============================================================================
# 4. STRUCTURE DU DATASET — VARIABLES CLÉS
# =============================================================================
 
print("\n" + "─" * 65)
print("STATISTIQUES CLÉS — VUE D'ENSEMBLE")
print("─" * 65)
 
n_interactions = len(df)
n_subjects     = df["iid"].nunique()
n_waves        = df["wave"].nunique()
match_rate     = df["match"].mean() * 100
match_rate_f   = df[df["gender"] == 0]["match"].mean() * 100
match_rate_m   = df[df["gender"] == 1]["match"].mean() * 100
n_female       = df[df["gender"] == 0]["iid"].nunique()
n_male         = df[df["gender"] == 1]["iid"].nunique()
 
print(f"\nInteractions totales  : {n_interactions:,}")
print(f"Participants uniques  : {n_subjects:,}")
print(f"> Femmes (gender=0) : {n_female}")
print(f"> Hommes (gender=1) : {n_male}")
print(f"\nVagues (events) : {n_waves}")
print(f"Taux de match global  : {match_rate:.1f}%")
print(f"Vu par les femmes : {match_rate_f:.1f}%")
print(f"Vu par les hommes : {match_rate_m:.1f}%")


─────────────────────────────────────────────────────────────────
STATISTIQUES CLÉS — VUE D'ENSEMBLE
─────────────────────────────────────────────────────────────────



Interactions totales  : 8,378
Participants uniques  : 551
> Femmes (gender=0) : 274
> Hommes (gender=1) : 277

Vagues (events) : 21
Taux de match global  : 16.5%
Vu par les femmes : 16.5%
Vu par les hommes : 16.5%


In [46]:
# =============================================================================
# 5. PROBLÈME CRITIQUE : NORMALISATION DES WAVES
# =============================================================================
 
print("\n" + "─" * 65)
print("PROBLÈME CRITIQUE — DEUX ÉCHELLES DE MESURE")
print("─" * 65)
 
WAVES_100PTS = [1, 2, 3, 4, 5, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
WAVES_1_10   = [6, 7, 8, 9]
 
pref_cols = ["attr1_1", "sinc1_1", "intel1_1", "fun1_1", "amb1_1", "shar1_1"]
 
mean_100 = df[df["wave"].isin(WAVES_100PTS)][pref_cols].mean()
mean_110 = df[df["wave"].isin(WAVES_1_10)][pref_cols].mean()
 
print("\nColonnes préférences déclarées (attr1_1, sinc1_1...) :")
print(f"Waves 1-5 & 10-21 → échelle 100 pts → moyenne attr1_1 : {mean_100['attr1_1']:.1f}")
print(f"Waves 6-9         → échelle 1-10    → moyenne attr1_1 : {mean_110['attr1_1']:.1f}")
print("\n> Incomparables sans normalisation ! Corriger avant toute analyse.")
print()
 
# Visualisation de l'impact du problème
fig_scale = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Avant normalisation (mélange des échelles)",
                    "Après normalisation (tout en 1-10)"]
)
 
# Avant : distribution brute de attr1_1
fig_scale.add_trace(
    go.Histogram(
        x=df["attr1_1"],
        nbinsx=30,
        name="Brut",
        marker_color=PALETTE["no_match"],
        opacity=0.75,
        showlegend=False
    ), row=1, col=1
)
 
# Après normalisation
df_norm = df.copy()
df_norm.loc[df_norm["wave"].isin(WAVES_100PTS), pref_cols] = (
    df_norm.loc[df_norm["wave"].isin(WAVES_100PTS), pref_cols] / 10
)
fig_scale.add_trace(
    go.Histogram(
        x=df_norm["attr1_1"],
        nbinsx=30,
        name="Normalisé",
        marker_color=PALETTE["match"],
        opacity=0.75,
        showlegend=False
    ), row=1, col=2
)
 
fig_scale.update_layout(
    title="Impact de la normalisation — attr1_1 (importance attractivité)",
    height=380,
    plot_bgcolor="white"
)
fig_scale.update_xaxes(title_text="Score brut", row=1, col=1)
fig_scale.update_xaxes(title_text="Score normalisé (1-10)", row=1, col=2)
fig_scale.show()


─────────────────────────────────────────────────────────────────
PROBLÈME CRITIQUE — DEUX ÉCHELLES DE MESURE
─────────────────────────────────────────────────────────────────

Colonnes préférences déclarées (attr1_1, sinc1_1...) :
Waves 1-5 & 10-21 → échelle 100 pts → moyenne attr1_1 : 24.0
Waves 6-9         → échelle 1-10    → moyenne attr1_1 : 16.2

> Incomparables sans normalisation ! Corriger avant toute analyse.



In [47]:
# =============================================================================
# 6. DISTRIBUTION DES WAVES (TAILLE DES EVENTS)
# =============================================================================
 
wave_counts = df["wave"].value_counts().sort_index().reset_index()
wave_counts.columns = ["Wave", "Nb interactions"]
wave_counts["Echelle"] = wave_counts["Wave"].apply(
    lambda w: "100 pts (÷10)" if w in WAVES_100PTS else "1-10 (ok)"
)
 
fig_waves = px.bar(
    wave_counts,
    x="Wave",
    y="Nb interactions",
    color="Echelle",
    color_discrete_map={"100 pts (÷10)": PALETTE["amber"], "1-10 (ok)": PALETTE["match"]},
    title="Nombre d'interactions par vague (event)",
    labels={"Wave": "Vague", "Nb interactions": "Nombre d'interactions"},
    height=380,
    text="Nb interactions",
)
fig_waves.update_traces(textposition="outside", textfont_size=10)
fig_waves.update_layout(
    xaxis=dict(tickmode="linear", dtick=1),
    plot_bgcolor="white",
    title_font_size=16,
)
fig_waves.show()

In [48]:
# =============================================================================
# 7. SÉLECTION DES COLONNES POUR L'ANALYSE PRINCIPALE
# =============================================================================
 
CORE_COLS = [
    "iid", "id", "gender", "wave", "condtn",
    "match",
    "dec", "attr", "sinc", "intel", "fun", "amb", "shar", "like", "prob",
    "dec_o", "attr_o", "sinc_o", "intel_o", "fun_o", "amb_o", "shar_o", "like_o", "prob_o",
    "attr1_1", "sinc1_1", "intel1_1", "fun1_1", "amb1_1", "shar1_1",
    "attr3_1", "sinc3_1", "fun3_1", "intel3_1", "amb3_1",
    "int_corr", "samerace", "age_o", "race_o",
    "age", "race", "field_cd", "goal", "go_out", "career_c", "income",
]
CORE_COLS = [c for c in CORE_COLS if c in df.columns]
 
df_clean = df[CORE_COLS].copy()
 
# Normalisation des préférences déclarées
pref_cols_clean = ["attr1_1", "sinc1_1", "intel1_1", "fun1_1", "amb1_1", "shar1_1"]
df_clean.loc[df_clean["wave"].isin(WAVES_100PTS), pref_cols_clean] = (
    df_clean.loc[df_clean["wave"].isin(WAVES_100PTS), pref_cols_clean] / 10
)
 
# Labels lisibles
df_clean["gender_label"] = df_clean["gender"].map({0: "Femme", 1: "Homme"})
df_clean["match_label"]  = df_clean["match"].map({0: "Pas de match", 1: "Match"})
 
print("─" * 65)
print("DATASET NETTOYÉ — PRÊT POUR L'EDA")
print("─" * 65)
print(f"\n  Colonnes sélectionnées : {len(CORE_COLS)} (sur 195)")
print(f"  Lignes                 : {len(df_clean):,}")
print(f"  Valeurs manquantes     :\n")
missing_core = df_clean[CORE_COLS].isnull().mean() * 100
missing_core = missing_core[missing_core > 0].sort_values(ascending=False)
for col, pct in missing_core.items():
    flag = "x " if pct > 10 else "v "
    print(f"    {flag}{col:<20} {pct:>5.1f}%")


─────────────────────────────────────────────────────────────────
DATASET NETTOYÉ — PRÊT POUR L'EDA
─────────────────────────────────────────────────────────────────

  Colonnes sélectionnées : 46 (sur 195)
  Lignes                 : 8,378
  Valeurs manquantes     :

    x income                48.9%
    x shar_o                12.8%
    x shar                  12.7%
    v amb_o                  8.6%
    v amb                    8.5%
    v fun_o                  4.3%
    v fun                    4.2%
    v prob_o                 3.8%
    v prob                   3.7%
    v intel_o                3.7%
    v intel                  3.5%
    v sinc_o                 3.4%
    v sinc                   3.3%
    v like_o                 3.0%
    v like                   2.9%
    v attr_o                 2.5%
    v attr                   2.4%
    v int_corr               1.9%
    v career_c               1.6%
    v shar1_1                1.4%
    v amb3_1                 1.3%
    v fun3_1      

In [49]:
# =============================================================================
# 8. APERÇU DU DATASET FINAL
# =============================================================================
 
print("\n" + "─" * 65)
print("APERÇU — 5 PREMIÈRES LIGNES DU DATASET NETTOYÉ")
print("─" * 65)
pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 120)
display_cols = [
    "iid", "gender_label", "wave", "match_label",
    "attr", "sinc", "fun", "like", "prob",
    "attr1_1", "sinc1_1", "fun1_1",
    "int_corr", "samerace"
]
display_cols = [c for c in display_cols if c in df_clean.columns]
print(df_clean[display_cols].head().to_string(index=False))


─────────────────────────────────────────────────────────────────
APERÇU — 5 PREMIÈRES LIGNES DU DATASET NETTOYÉ
─────────────────────────────────────────────────────────────────
 iid gender_label  wave  match_label  attr  sinc  fun  like  prob  attr1_1  sinc1_1  fun1_1  int_corr  samerace
   1        Femme     1 Pas de match   6.0   9.0  7.0   7.0   6.0      1.5      2.0     1.5      0.14         0
   1        Femme     1 Pas de match   7.0   8.0  8.0   7.0   5.0      1.5      2.0     1.5      0.54         0
   1        Femme     1        Match   5.0   8.0  8.0   7.0   NaN      1.5      2.0     1.5      0.16         1
   1        Femme     1        Match   7.0   6.0  7.0   7.0   6.0      1.5      2.0     1.5      0.61         0
   1        Femme     1        Match   5.0   6.0  7.0   6.0   6.0      1.5      2.0     1.5      0.21         0


In [50]:
# =============================================================================
# 9. VISUALISATION SYNTHÈSE — DASHBOARD COMPRÉHENSION
# =============================================================================
 
fig_dash = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Distribution du match (target)",
        "Répartition Hommes / Femmes",
        "Taille des vagues dans le temps",
        "Qualité données — colonnes clés"
    ],
    specs=[[{"type": "pie"}, {"type": "pie"}],
           [{"type": "bar"}, {"type": "bar"}]],
    vertical_spacing=0.18,
    horizontal_spacing=0.12,
)
 
# Pie 1 — match
match_counts = df_clean["match_label"].value_counts()
fig_dash.add_trace(
    go.Pie(
        labels=match_counts.index,
        values=match_counts.values,
        marker_colors=[PALETTE["match"], PALETTE["no_match"]],
        hole=0.45,
        textinfo="label+percent",
        showlegend=False,
    ), row=1, col=1
)
 
# Pie 2 — genre
gender_counts = df_clean["gender_label"].value_counts()
fig_dash.add_trace(
    go.Pie(
        labels=gender_counts.index,
        values=gender_counts.values,
        marker_colors=[PALETTE["female"], PALETTE["male"]],
        hole=0.45,
        textinfo="label+percent",
        showlegend=False,
    ), row=1, col=2
)
 
# Bar — taille waves
fig_dash.add_trace(
    go.Bar(
        x=wave_counts["Wave"],
        y=wave_counts["Nb interactions"],
        marker_color=wave_counts["Echelle"].map({
            "100 pts (÷10)": PALETTE["amber"],
            "1-10 (ok)": PALETTE["match"]
        }),
        showlegend=False,
    ), row=2, col=1
)
 
# Bar — missing colonnes core
top_missing = missing_core.head(15)
fig_dash.add_trace(
    go.Bar(
        x=top_missing.index,
        y=top_missing.values,
        marker_color=[
            PALETTE["no_match"] if v > 30 else
            PALETTE["amber"] if v > 5 else
            PALETTE["match"]
            for v in top_missing.values
        ],
        showlegend=False,
    ), row=2, col=2
)
 
fig_dash.update_layout(
    title_text="PROJET TINDER — Dashboard compréhension dataset",
    title_font_size=18,
    height=620,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig_dash.update_xaxes(tickangle=-45, row=2, col=2)
fig_dash.update_yaxes(title_text="Nb interactions", row=2, col=1)
fig_dash.update_yaxes(title_text="% manquant", row=2, col=2)
 
fig_dash.show()

In [52]:
# =============================================================================
# 10. EXPORT DU DATASET NETTOYÉ
# =============================================================================
 
df_clean.to_csv(f"{OUTPUT_DATA_PATH}/df_speed_dating_prep.csv", index=False)
 
print(f"\nDataset nettoyé exporté : {OUTPUT_DATA_PATH}/df_speed_dating_prep.csv")
print(f"\nRÉSUMÉ :")
print(f"- 8 378 interactions, 16,5% de matchs (classe déséquilibrée)")
print(f"- Colonnes utiles sélectionnées : {len(CORE_COLS)}/195")
print(f"- Normalisation waves appliquée (÷10 pour waves 1-5 & 10-21)")
print(f"- T2/T3 exclus (>76% manquants)")



Dataset nettoyé exporté : ../outputs/data/df_speed_dating_prep.csv

RÉSUMÉ :
- 8 378 interactions, 16,5% de matchs (classe déséquilibrée)
- Colonnes utiles sélectionnées : 46/195
- Normalisation waves appliquée (÷10 pour waves 1-5 & 10-21)
- T2/T3 exclus (>76% manquants)
